# Fitur Metadata Dataset Pemodelan

Notebook ini membangun representasi metadata untuk 387.188 film pada dataset
pemodelan, lalu memeriksa hasilnya secara kualitatif.

Fitur metadata dibentuk dari empat kolom: `genres`, `directors`, `writers`, dan
`cast`. Keempatnya berisi daftar label yang dipisahkan koma, bukan kalimat.

Karena sifatnya daftar label, representasi yang dipakai adalah bag of words dengan
CountVectorizer, bukan SBERT. Setiap nilai metadata diperlakukan sebagai token
diskrit dengan prefiks peran, sehingga kecocokan bersifat eksak. Pendekatan ini
berbeda dari jalur teks semantik di notebook 03, yang memerlukan SBERT untuk
menangkap kemiripan makna.

Notebook ini menghasilkan lima artifact di `artifacts/metadata/`, ditambah satu
tabel rekomendasi untuk pemeriksaan kualitatif.

Yang tidak termasuk di notebook ini: cosine similarity untuk seluruh film, fusion
dengan parameter `alpha`, recommendation engine, dan metrik evaluasi.


## 1. Tujuan dan Ruang Lingkup

Notebook ini menjawab satu pertanyaan: bagaimana merepresentasikan metadata film
sebagai vektor numerik yang bisa dibandingkan dengan cosine similarity.

Ada tiga syarat yang harus dipenuhi hasilnya.

Pertama, setiap film harus menghasilkan vektor yang tidak nol. Kalau ada film tanpa
token, normanya nol, dan cosine similarity terhadapnya tidak terdefinisi.

Kedua, urutan baris harus sama dengan urutan embedding semantik. Tanpa itu,
penggabungan dua jalur fitur tidak bisa dilakukan berdasarkan posisi baris.

Ketiga, skala skornya harus sebanding dengan jalur semantik. Kalau jalur metadata
hanya bisa mencapai skor 0,05 sementara semantik mencapai 0,7, nilai `alpha` pada
fusion kehilangan makna.

Bagian akhir notebook memeriksa satu hal yang tidak termasuk tiga syarat di atas,
yaitu apakah hasil rekomendasi metadata masuk akal secara isi. Pemeriksaan ini
bukan evaluasi formal, melainkan sanity check.


## 2. Konfigurasi dan Pemetaan Peran

Empat kolom metadata dipetakan ke prefiks peran yang berbeda:

| Kolom | Prefiks | Rata-rata item per film |
|---|---|---:|
| `genres` | `genre_` | 1,71 |
| `directors` | `director_` | 1,16 |
| `writers` | `writer_` | 1,88 |
| `cast` | `cast_` | 7,28 |

Prefiks berfungsi untuk dua hal. Pertama, mencegah tabrakan antara nama orang dan
nama genre yang kebetulan sama. Kedua, memungkinkan analisis kontribusi per peran
setelah vektorisasi, karena setiap fitur bisa dilacak asalnya.

`CAST_LIMIT` dibatasi 5. Alasannya, `cast` punya rata-rata 7,28 nama per film,
sementara `directors` hanya 1,16. Kalau semua nama dipakai, aktor akan mendominasi
vektor bukan karena keputusan desain, tetapi karena jumlahnya.

`MERGE_PREFIXES` menangani masalah koma di dalam nama orang. Nilai metadata
dipisahkan koma, tetapi sebagian nama juga mengandung koma, misalnya
`Sophie, Duchess of Edinburgh`. Tanpa aturan penggabungan, satu orang akan terpecah
menjadi dua token.


In [2]:
import json
import re

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize

from movie_recommender.utils.paths import (
    ARTIFACTS_DIR,
    DATA_PROCESSED_DIR,
    SEMANTIC_ARTIFACTS_DIR,
)

MODELING_PARQUET = DATA_PROCESSED_DIR / "movies_modeling.parquet"
METADATA_DIR = ARTIFACTS_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

ROLE_COLUMNS = {
    "genres": "genre",
    "directors": "director",
    "writers": "writer",
    "cast": "cast",
}
CAST_LIMIT = 5

# Segmen yang diawali kata ini digabung ke segmen sebelumnya,
# karena koma di dalam nama orang bukan pemisah.
MERGE_PREFIXES = {
    "jr", "jr.", "sr", "sr.", "ii", "iii", "iv", "v",
    "duke", "duchess", "count", "countess",
    "baron", "baroness", "earl", "lady", "lord",
    "sir", "dame", "president", "prince", "princess",
}

Cell di atas memuat seluruh konfigurasi dan membuat folder artifact.

Perhatikan bahwa `SEMANTIC_ARTIFACTS_DIR` diimpor walaupun belum dipakai. Import ini
dibutuhkan pada cell verifikasi nanti, untuk memastikan urutan ID metadata sama
dengan urutan ID embedding.

`ROLE_COLUMNS` sengaja ditulis sebagai dictionary, bukan list, supaya pemetaan antara
kolom sumber dan prefiks peran tersimpan di satu tempat. Dengan begitu, penambahan
kolom baru di masa depan cukup mengubah satu baris.


## 3. Parsing Nilai Metadata

Tiga fungsi di cell berikut menangani seluruh proses parsing.

`split_items` memecah string berdasarkan koma, lalu menggabungkan kembali segmen yang
sebenarnya bagian dari nama yang sama. Caranya dengan memeriksa kata pertama setiap
segmen. Kalau kata pertamanya ada di `MERGE_PREFIXES`, segmen itu digabung ke segmen
sebelumnya.

Contoh nyata dari dataset:

```text
Sophie, Duchess of Edinburgh
```

Tanpa aturan penggabungan, hasilnya dua item: `Sophie` dan `Duchess of Edinburgh`.
Dengan aturan penggabungan, hasilnya satu item, dan `Duchess` ada di daftar prefiks
sehingga segmen kedua digabung kembali.

`slug` mengubah teks menjadi bentuk aman untuk token. Karakter selain huruf dan angka
diganti garis bawah, lalu garis bawah di ujung dibuang. Fungsinya agar
`Christopher Nolan`, `christopher nolan`, dan `Christopher-Nolan` menghasilkan token
yang sama.

`build_metadata_text` menggabungkan keduanya. Untuk setiap film, fungsi ini membaca
empat kolom, memecah nilainya, membatasi `cast` jika perlu, lalu menempelkan prefiks
peran pada setiap item.


In [3]:
def split_items(value: object) -> list[str]:
    if not isinstance(value, str) or not value.strip():
        return []

    parts = [part.strip() for part in value.split(",")]
    parts = [part for part in parts if part]

    merged: list[str] = []
    for part in parts:
        words = part.split()
        if not words:
            continue
        head = words[0].casefold()
        if merged and head in MERGE_PREFIXES:
            merged[-1] = f"{merged[-1]}, {part}"
        else:
            merged.append(part)

    return merged


def slug(value: str) -> str:
    cleaned = re.sub(r"[^0-9a-z]+", "_", value.casefold())
    return cleaned.strip("_")


def build_metadata_text(movies: pd.DataFrame) -> pd.Series:
    tokens_per_film: list[list[str]] = []

    for row in movies[list(ROLE_COLUMNS)].itertuples(index=False):
        tokens: list[str] = []

        for value, prefix in zip(row, ROLE_COLUMNS.values()):
            items = split_items(value)

            if prefix == "cast":
                items = items[:CAST_LIMIT]

            for item in items:
                token = slug(item)
                if token:
                    tokens.append(f"{prefix}_{token}")

        tokens_per_film.append(tokens)

    return pd.Series(
        [" ".join(tokens) for tokens in tokens_per_film],
        index=movies.index,
        name="metadata_text",
        dtype="string",
    )

Satu hal yang perlu diperhatikan: `split_items` dan `slug` bekerja per item, bukan per
film. Untuk 387.188 film dengan rata-rata 7,86 token, ini berarti sekitar tiga juta
pemanggilan fungsi. Prosesnya masih cepat karena setiap panggilan hanya melakukan
operasi string sederhana.

Kalau nanti dataset bertambah besar secara signifikan, bagian ini kandidat pertama
untuk divektorisasi dengan pandas string operations. Tapi untuk ukuran sekarang,
loop eksplisit lebih mudah dibaca dan di-debug.


## 4. Membentuk `metadata_text`

Cell berikut menjalankan parsing untuk seluruh dataset, lalu memeriksa dua hal:
jumlah film dan jumlah `metadata_text` yang kosong.

Pemeriksaan kosong ini penting karena menjadi gerbang sebelum vektorisasi. Kalau ada
satu film saja yang tidak menghasilkan token, barisnya akan bernorma nol, dan cosine
similarity terhadap film itu tidak terdefinisi.


In [4]:
movies = pd.read_parquet(MODELING_PARQUET).reset_index(drop=True)

print("film:", len(movies))
print("ID unik:", movies["id"].is_unique)

metadata_text = build_metadata_text(movies)

empty = metadata_text.str.strip().eq("")
print("metadata_text kosong:", int(empty.sum()))

if empty.any():
    raise ValueError(
        f"{int(empty.sum())} film tidak menghasilkan token metadata"
    )

print("contoh:")
for index in range(3):
    print(" ", movies.loc[index, "title"], "->", metadata_text.loc[index][:150])

film: 387188
ID unik: True
metadata_text kosong: 0
contoh:
  Inception -> genre_action genre_science_fiction genre_adventure director_christopher_nolan writer_christopher_nolan cast_leonardo_dicaprio cast_joseph_gordon_levit
  Interstellar -> genre_adventure genre_drama genre_science_fiction director_christopher_nolan writer_jonathan_nolan writer_christopher_nolan cast_matthew_mcconaughey c
  The Dark Knight -> genre_drama genre_action genre_crime genre_thriller director_christopher_nolan writer_jonathan_nolan writer_christopher_nolan writer_david_s_goyer wri


Hasilnya 387.188 film, seluruh ID unik, dan nol `metadata_text` kosong.

Ini konsekuensi dari filter di notebook 01 bagian 13, yang membuang 1.053 film tanpa
metadata. Tanpa filter itu, akan ada 1.053 baris bernorma nol di sini.

Tiga contoh di output memperlihatkan bentuk teks yang dihasilkan. Perhatikan bahwa
setiap item membawa prefiks peran, dan nama orang sudah diubah ke bentuk slug dengan
garis bawah sebagai pemisah.


## 5. Vektorisasi dengan CountVectorizer

Cell berikut mengubah `metadata_text` menjadi matriks sparse.

Pengaturan yang dipakai dan alasannya:

| Parameter | Nilai | Alasan |
|---|---|---|
| `binary` | `True` | Metadata adalah keanggotaan himpunan, bukan frekuensi |
| `lowercase` | `False` | Teks sudah di-slug, jadi sudah huruf kecil semua |
| `token_pattern` | `\b\w+\b` | Default sklearn membuang token satu karakter |
| `min_df` | `1` | Nama yang muncul sekali tetap dipertahankan |
| `dtype` | `float32` | Hemat memori, konsisten dengan embedding semantik |

`binary=True` dipilih karena hampir semua token metadata muncul tepat satu kali per
film. Dari pemeriksaan terpisah, 99,5 persen film tidak punya token duplikat. Dengan
`binary=True`, nilai sel selalu 0 atau 1, dan cara ini sekaligus mengatasi kasus
langka ketika satu nama tercatat dua kali.

`min_df=1` dipilih walaupun menghasilkan kosakata besar. Kalau diset 2, nama yang
hanya muncul di satu film akan dibuang. Itu justru merugikan, karena film langka
tidak akan punya sinyal sama sekali.

Setelah vektorisasi, matriks dinormalisasi L2 per baris. Tujuannya agar cosine
similarity cukup dihitung sebagai dot product, sama seperti jalur semantik.


In [5]:
vectorizer = CountVectorizer(
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b",
    min_df=1,
    binary=True,
    dtype=np.float32,
)

matrix = vectorizer.fit_transform(metadata_text.tolist())
print("bentuk mentah:", matrix.shape)
print("nilai bukan nol:", matrix.nnz)

matrix = normalize(matrix, norm="l2", axis=1, copy=False)

nnz_per_row = np.diff(matrix.indptr)
print("film dengan 0 token:", int((nnz_per_row == 0).sum()))
print("token per film min/median/max:",
      int(nnz_per_row.min()),
      int(np.median(nnz_per_row)),
      int(nnz_per_row.max()))

bentuk mentah: (387188, 841843)
nilai bukan nol: 3044037
film dengan 0 token: 0
token per film min/median/max: 1 8 575


Hasilnya matriks 387.188 x 841.843 dengan 3.044.037 nilai bukan nol.

Kalau dihitung, kepadatannya sekitar 9,34 kali 10 pangkat -6. Artinya 99,999 persen
sel matriks bernilai nol. Inilah alasan format sparse dipakai: kalau disimpan sebagai
matriks penuh, ukurannya sekitar 1,3 TB.

Token per film: minimum 1, median 8, maksimum 575.

Nilai maksimum 575 perlu dijelaskan. Itu bukan kesalahan parsing, melainkan film
antologi atau dokumenter kolaboratif yang memang melibatkan puluhan sutradara. Contoh
terbesarnya `Lebanon, at the heart of chaos` dengan 404 nama sutradara.

Sebaran token yang sangat miring ini akan dibahas di bagian terakhir, karena
mempengaruhi cara skor similarity terbentuk.


## 6. Menyimpan Artifact

Cell berikut menyimpan lima artifact ke `artifacts/metadata/`.

| File | Isi |
|---|---|
| `metadata_matrix.npz` | Matriks sparse yang sudah dinormalisasi L2 |
| `metadata_text.parquet` | `id` dan teks metadata per film |
| `movie_ids.npy` | Urutan ID, acuan penggabungan dengan embedding |
| `vectorizer.joblib` | CountVectorizer yang sudah dilatih |
| `config.json` | Parameter, jumlah fitur, dan kepadatan matriks |

`vectorizer.joblib` disimpan walaupun strukturnya besar, sekitar 23 MB. Alasannya,
kalau nanti ada film baru, kosakatanya harus dipakai ulang, bukan dilatih ulang.
Melatih ulang akan menggeser indeks kolom, dan matriks lama jadi tidak kompatibel.

`config.json` mencatat seluruh keputusan yang mempengaruhi hasil, termasuk daftar
`MERGE_PREFIXES`. Tanpa catatan itu, orang lain tidak bisa mereproduksi matriks yang
sama persis, karena aturan penggabungan nama adalah keputusan yang tidak terlihat
dari data mentah.


In [7]:
movie_ids = movies["id"].to_numpy(dtype=np.int64)

sparse.save_npz(METADATA_DIR / "metadata_matrix.npz", matrix)
np.save(METADATA_DIR / "movie_ids.npy", movie_ids)
joblib.dump(vectorizer, METADATA_DIR / "vectorizer.joblib")

pd.DataFrame(
    {
        "id": movie_ids,
        "metadata_text": pd.array(metadata_text.to_numpy(), dtype="string"),
    }
).to_parquet(
    METADATA_DIR / "metadata_text.parquet",
    index=False,
    compression="zstd",
)

config = {
    "build_method": "countvectorizer_binary_l2",
    "input_dataset": "data/processed/movies_modeling.parquet",
    "role_columns": ROLE_COLUMNS,
    "cast_limit": CAST_LIMIT,
    "merge_prefixes": sorted(MERGE_PREFIXES),
    "vectorizer": "CountVectorizer",
    "binary": True,
    "lowercase": False,
    "token_pattern": r"(?u)\b\w+\b",
    "min_df": 1,
    "normalization": "l2",
    "n_films": int(matrix.shape[0]),
    "n_features": int(matrix.shape[1]),
    "nnz": int(matrix.nnz),
    "density": round(matrix.nnz / (matrix.shape[0] * matrix.shape[1]), 8),
}
(METADATA_DIR / "config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

print(json.dumps(config, indent=2))

{
  "build_method": "countvectorizer_binary_l2",
  "input_dataset": "data/processed/movies_modeling.parquet",
  "role_columns": {
    "genres": "genre",
    "directors": "director",
    "writers": "writer",
    "cast": "cast"
  },
  "cast_limit": 5,
  "merge_prefixes": [
    "baron",
    "baroness",
    "count",
    "countess",
    "dame",
    "duchess",
    "duke",
    "earl",
    "ii",
    "iii",
    "iv",
    "jr",
    "jr.",
    "lady",
    "lord",
    "president",
    "prince",
    "princess",
    "sir",
    "sr",
    "sr.",
    "v"
  ],
  "vectorizer": "CountVectorizer",
  "binary": true,
  "lowercase": false,
  "token_pattern": "(?u)\\b\\w+\\b",
  "min_df": 1,
  "normalization": "l2",
  "n_films": 387188,
  "n_features": 841843,
  "nnz": 3044037,
  "density": 9.34e-06
}


Cell di atas menulis lima file. Perhatikan bahwa `matrix` yang disimpan adalah hasil
dari cell vektorisasi, sedangkan `movie_ids` diambil dari kolom `id` dataset
pemodelan. Keduanya berasal dari sumber yang sama, sehingga urutannya konsisten.


## 7. Verifikasi Artifact

Verifikasi membaca kembali file dari disk, bukan memakai variabel di memori. Tujuannya
memastikan artifact yang tersimpan benar-benar bisa dipakai pada tahap berikutnya.

Ada empat pemeriksaan yang paling penting.

**Urutan ID sama dengan embedding.** Ini syarat mutlak untuk penggabungan dua jalur
fitur. Kalau urutannya berbeda, film A akan mendapat metadata film B, dan kesalahannya
tidak akan terlihat sampai hasil rekomendasi diperiksa.

**Norma setiap baris sama dengan 1.** Kalau ada baris bernorma 0, cosine similarity
terhadap baris itu tidak terdefinisi, dan hasil fusion akan menghasilkan `NaN`.

**Tidak ada nilai non-finite.** `NaN` atau `inf` di dalam matriks akan merambat ke
seluruh perhitungan setelahnya.

**Jumlah fitur dan jumlah baris sesuai config.** Ini memastikan `config.json` benar
benar menggambarkan artifact, bukan hasil percobaan sebelumnya.


In [8]:
matrix_check = sparse.load_npz(METADATA_DIR / "metadata_matrix.npz")
ids_check = np.load(METADATA_DIR / "movie_ids.npy")
semantic_ids = np.load(SEMANTIC_ARTIFACTS_DIR / "movie_ids.npy")
config_check = json.loads(
    (METADATA_DIR / "config.json").read_text(encoding="utf-8")
)

norms = np.sqrt(matrix_check.multiply(matrix_check).sum(axis=1)).A1

checks = {
    "shape": matrix_check.shape,
    "tipe": type(matrix_check).__name__,
    "urutan ID sama dengan embedding": bool(
        np.array_equal(ids_check, semantic_ids)
    ),
    "ID unik": len(np.unique(ids_check)) == len(ids_check),
    "n_features sesuai config": config_check["n_features"] == matrix_check.shape[1],
    "n_films sesuai config": config_check["n_films"] == matrix_check.shape[0],
    "norm minimum": float(norms.min()),
    "norm maksimum": float(norms.max()),
    "baris dengan norm 0": int((norms == 0).sum()),
    "nilai non-finite": int((~np.isfinite(matrix_check.data)).sum()) if matrix_check.nnz else 0,
}

pd.Series(checks, name="value")

shape                              (387188, 841843)
tipe                                     csr_matrix
urutan ID sama dengan embedding                True
ID unik                                        True
n_features sesuai config                       True
n_films sesuai config                          True
norm minimum                                    1.0
norm maksimum                                   1.0
baris dengan norm 0                               0
nilai non-finite                                  0
Name: value, dtype: object

Seluruh pemeriksaan lulus. Matriks berbentuk 387.188 x 841.843, bertipe `csr_matrix`,
urutan ID sama dengan embedding, norm minimum dan maksimum 1,0, tidak ada baris
bernorma nol, dan tidak ada nilai non-finite.

Dengan begitu, tiga syarat di bagian 1 sudah terpenuhi. Artifact metadata siap
dipakai untuk perhitungan similarity.

Syarat ketiga, yaitu kesebandingan skala dengan jalur semantik, baru bisa diperiksa
setelah similarity dihitung. Itu dilakukan di bagian 8.


## 8. Spot Check Rekomendasi Metadata

Bagian ini memeriksa hasil rekomendasi dari jalur metadata saja, tanpa campuran fitur
semantik. Tujuannya bukan evaluasi formal, melainkan pemeriksaan kewajaran: apakah
film yang muncul memang berhubungan secara metadata.

Matriks metadata sudah dinormalisasi L2, sehingga cosine similarity setara dengan dot
product. Karena matriksnya sparse, perhitungan dilakukan dengan operasi sparse, bukan
mengubah seluruh matriks menjadi dense.

Kolom `genre_overlap` adalah Jaccard similarity antar himpunan genre. Nilai 1.00
berarti genrenya persis sama. Kolom ini hanya alat bantu interpretasi dan tidak dipakai
untuk ranking.

Satu hal yang perlu diperhatikan saat membaca hasil: skor tinggi pada jalur metadata
bisa berarti dua hal yang berbeda. Pertama, film memang dibuat oleh orang yang sama.
Kedua, film hanya kebetulan punya kombinasi genre yang sama. Keduanya menghasilkan
angka tinggi, dan hanya kolom `genre_overlap` serta `directors` yang bisa
membedakannya.


In [9]:
import numpy as np
import pandas as pd
from scipy import sparse

from movie_recommender.utils.paths import ARTIFACTS_DIR, DATA_PROCESSED_DIR

METADATA_DIR = ARTIFACTS_DIR / "metadata"

metadata_matrix = sparse.load_npz(
    METADATA_DIR / "metadata_matrix.npz"
).tocsr()
metadata_ids = np.load(METADATA_DIR / "movie_ids.npy")

movies = pd.read_parquet(
    DATA_PROCESSED_DIR / "movies_modeling.parquet",
    columns=[
        "id",
        "title",
        "release_date",
        "genres",
        "directors",
        "vote_average",
    ],
).set_index("id")

row_of = pd.Series(np.arange(len(metadata_ids)), index=metadata_ids)

print("matrix:", metadata_matrix.shape, metadata_matrix.dtype)
print("nnz:", metadata_matrix.nnz)
print("film:", len(movies))
print("urutan ID cocok:", np.array_equal(metadata_ids, movies.index.to_numpy()))

matrix: (387188, 841843) float32
nnz: 3044037
film: 387188
urutan ID cocok: True


Cell di atas memuat matriks, ID, dan metadata film, lalu membuat pemetaan dari ID ke
posisi baris.

Baris terakhir adalah pemeriksaan yang paling penting: `urutan ID cocok` harus
bernilai `True`. Kalau `False`, posisi baris tidak sejajar dengan dataset, dan semua
hasil query akan menunjuk film yang salah.

Perhatikan juga bahwa `movies` di sini hanya memuat enam kolom. Kolom `overview`,
`writers`, dan `cast` tidak dimuat, karena tidak dibutuhkan untuk menampilkan hasil.


### 8.1 Fungsi Pencarian Top-K

Tiga fungsi di cell berikut menangani seluruh proses pencarian.

`movie_year` mengekstrak tahun dari tanggal rilis. Nilai yang tidak bisa di-parse
dikembalikan sebagai `None`, bukan error, karena sebagian film punya tanggal rilis
kosong.

`genre_overlap` menghitung Jaccard similarity antara dua himpunan genre. Nilainya
0 sampai 1, dan hanya dipakai sebagai alat bantu membaca hasil.

`metadata_top_k` menghitung skor satu film terhadap seluruh film lain, mengeluarkan
film itu sendiri, lalu mengambil K teratas. Hasilnya digabungkan dengan judul, tahun,
genre, sutradara, dan rating.


In [10]:
def movie_year(value):
    date = pd.to_datetime(value, errors="coerce")
    return None if pd.isna(date) else int(date.year)


def genre_overlap(genres_a, genres_b) -> float:
    if pd.isna(genres_a) or pd.isna(genres_b):
        return 0.0

    set_a = {genre.strip() for genre in str(genres_a).split(",") if genre.strip()}
    set_b = {genre.strip() for genre in str(genres_b).split(",") if genre.strip()}

    union = set_a | set_b
    return len(set_a & set_b) / len(union) if union else 0.0


def metadata_top_k(movie_id: int, k: int = 10) -> pd.DataFrame:
    row = int(row_of.loc[movie_id])

    scores = np.asarray(
        (metadata_matrix @ metadata_matrix[row].T).todense()
    ).ravel().astype(np.float32)

    # Keluarkan film query dari kandidat
    scores[row] = -np.inf

    top_rows = np.argpartition(scores, -k)[-k:]
    top_rows = top_rows[np.argsort(scores[top_rows])[::-1]]
    top_ids = metadata_ids[top_rows]

    result = movies.loc[
        top_ids,
        ["title", "release_date", "genres", "directors", "vote_average"],
    ].copy()

    result.insert(0, "rank", range(1, len(result) + 1))
    result.insert(1, "recommended_movie_id", top_ids)
    result["similarity"] = scores[top_rows]
    result["year"] = result["release_date"].apply(movie_year)
    result["genre_overlap"] = [
        genre_overlap(movies.loc[movie_id, "genres"], genres)
        for genres in result["genres"]
    ]

    return result.drop(columns=["release_date"]).reset_index(drop=True)


def show_metadata_query(movie_id: int, k: int = 10) -> None:
    profile = movies.loc[movie_id]

    print("=" * 96)
    print(f"QUERY     : {profile['title']} ({movie_year(profile['release_date'])}) | id={movie_id}")
    print(f"Genres    : {profile['genres']}")
    print(f"Directors : {profile['directors']}")
    print("-" * 96)

    table = metadata_top_k(movie_id, k)
    print(
        table[
            ["rank", "title", "year", "similarity", "genre_overlap"]
        ].to_string(index=False)
    )
    print()

### 8.2 Menjalankan Contoh Query

Lima film dipakai sebagai sampel. Pemilihannya sama dengan notebook 03, supaya hasil
kedua jalur bisa dibandingkan secara langsung.


In [11]:
QUERY_IDS = [155, 27205, 862, 157336, 603]

for query_id in QUERY_IDS:
    show_metadata_query(query_id, k=10)

QUERY     : The Dark Knight (2008) | id=155
Genres    : Drama, Action, Crime, Thriller
Directors : Christopher Nolan
------------------------------------------------------------------------------------------------
 rank                          title  year  similarity  genre_overlap
    1          The Dark Knight Rises  2012    0.714286       1.000000
    2                  Batman Begins  2005    0.694365       0.750000
    3                   The Prestige  2006    0.462910       0.166667
    4            A Thought Unchained  2014    0.436436       1.000000
    5            Gangster Vs State 2  2021    0.404061       1.000000
    6                   Interstellar  2014    0.402911       0.166667
    7                    Harsh Times  2005    0.402911       1.000000
    8                    Harry Brown  2009    0.402911       1.000000
    9                     Rathaavara  2015    0.400892       0.750000
   10 Capture of the Biddle Brothers  1902    0.400892       0.750000

QUERY     : Inc

Pola yang muncul dari lima query di atas perlu dibaca dengan hati-hati, karena tidak
semuanya menunjukkan hal yang sama.

**The Dark Knight dan Toy Story bekerja sebagaimana mestinya.** The Dark Knight
menghasilkan The Dark Knight Rises dan Batman Begins, keduanya film Nolan dengan genre
serupa. Toy Story menghasilkan sekuel resmi plus beberapa film Pixar dengan sutradara
yang sama.

**The Matrix sangat kuat.** Dua peringkat teratas adalah The Matrix Reloaded dan The
Matrix Revolutions, keduanya disutradarai Wachowski bersaudara dengan genre yang
hampir identik.

**Inception justru menunjukkan kelemahan.** Empat peringkat teratas bukan film Nolan
sama sekali. Keempatnya film yang kebetulan punya tiga genre persis sama, yaitu
Action, Science Fiction, dan Adventure. Christopher Nolan baru muncul di peringkat 5
lewat The Dark Knight Rises.

Ini bukan kesalahan perhitungan. Ini konsekuensi representasi seragam yang akan
dibahas di bagian 10.


### 8.3 Statistik Distribusi Skor

Tabel berikut menjawab pertanyaan yang dibutuhkan untuk fusion: seberapa besar skor
yang bisa dicapai jalur metadata, dan bagaimana sebarannya.

Empat kolom yang diukur:

| Kolom | Arti |
|---|---|
| `top1` | Skor tertinggi untuk film itu |
| `top10` | Skor peringkat ke-10, batas bawah hasil rekomendasi |
| `rata_rata_semua_pasangan` | Rata-rata skor terhadap seluruh 387.188 film |
| `persen_skor_nol` | Persentase film lain yang tidak punya token sama sekali |
| `pasangan_nonzero` | Jumlah film yang punya minimal satu token sama |


In [12]:
QUERY_IDS = [27205, 157336, 155, 862, 603]
k = 10

rows = []

for query_id in QUERY_IDS:
    row = int(row_of.loc[query_id])

    scores = np.asarray(
        (metadata_matrix @ metadata_matrix[row].T).todense()
    ).ravel().astype(np.float32)
    scores[row] = -np.inf
    finite = scores[np.isfinite(scores)]

    top = np.sort(finite)[-k:][::-1]

    rows.append(
        {
            "film": movies.loc[query_id, "title"],
            "top1": round(float(top[0]), 4),
            "top10": round(float(top[-1]), 4),
            "rata_rata_semua_pasangan": round(float(finite.mean()), 4),
            "persen_skor_nol": round(float((finite == 0).mean() * 100), 2),
            "pasangan_nonzero": int((finite > 0).sum()),
        }
    )

pd.DataFrame(rows).set_index("film")

,top1,top10,rata_rata_semua_pasangan,persen_skor_nol,pasangan_nonzero
film,,,,,
Inception,0.4243,0.3873,0.0148,87.74,47455
Interstellar,0.5222,0.3636,0.0403,61.88,147611
The Dark Knight,0.7143,0.4009,0.0475,56.72,167580
Toy Story,0.5407,0.4588,0.0279,70.39,114661
The Matrix,0.8362,0.4558,0.0105,90.06,38480


Skor top-10 jalur metadata berada di 0,36 sampai 0,46. Ini sebanding dengan jalur
semantik yang berada di 0,52 sampai 0,65. Selisihnya sekitar 0,1 sampai 0,2, masih
dalam rentang yang wajar untuk fusion berbobot.

Temuan ini menjawab syarat ketiga di bagian 1: skala kedua jalur cukup sebanding,
sehingga `alpha` bisa dibaca sebagai bobot relatif, bukan sebagai proporsi kontribusi
yang harus dijumlahkan tepat 1.

Rata-rata skor seluruh pasangan hanya 0,01 sampai 0,05. Ini menunjukkan distribusi
yang sangat miring: hanya sebagian kecil pasangan yang punya kemiripan metadata.

Persentase skor nol berbeda jauh antar film, dari 56,72 persen sampai 90,06 persen.
The Matrix punya 90,06 persen pasangan bernilai nol, karena tokennya sangat spesifik.
Sebaliknya The Dark Knight hanya 56,72 persen, karena kombinasinya genre lebih umum.

Ini artinya jalur metadata sangat selektif. Bagi sebagian besar film, skor terhadapnya
nol, dan hanya sebagian kecil kandidat yang benar-benar punya sinyal. Sifat ini justru
menguntungkan untuk hybrid, karena metadata memberi informasi yang berbeda dari teks.


### 8.4 Menyimpan Hasil ke CSV

Cell berikut menyimpan 50 baris hasil, yaitu 10 rekomendasi untuk masing-masing lima
film query.

Filenya bernama `metadata_top10_descriptive.csv`, sejajar dengan
`semantic_top10_descriptive.csv` di notebook 03. Dengan begitu, kedua jalur bisa
ditampilkan berdampingan saat bimbingan.


In [13]:
from movie_recommender.utils.paths import RESULTS_DIR

recommendation_dir = RESULTS_DIR / "recommendations"
recommendation_dir.mkdir(parents=True, exist_ok=True)

rows = []

for query_id in QUERY_IDS:
    profile = movies.loc[query_id]
    table = metadata_top_k(query_id, k=10)

    table.insert(0, "query_movie_id", query_id)
    table.insert(1, "query_title", profile["title"])
    table.insert(2, "query_year", movie_year(profile["release_date"]))

    rows.append(table)

descriptive_path = recommendation_dir / "metadata_top10_descriptive.csv"
pd.concat(rows, ignore_index=True).to_csv(descriptive_path, index=False)

print("tersimpan:", descriptive_path)
print("baris:", sum(len(r) for r in rows))

tersimpan: D:\College Life\Skripsiw\hybrid-movie-recommender\results\recommendations\metadata_top10_descriptive.csv
baris: 50


## 9. Analisis Kontribusi Per Peran

Bagian ini menjawab satu pertanyaan yang muncul dari hasil spot check: kenapa film
dengan genre sama bisa mengalahkan film dengan sutradara sama.

Jawabannya ada di komposisi vektor. Matriks sudah dinormalisasi L2, sehingga jumlah
bobot kuadrat setiap film tepat 1. Kalau bobot kuadrat dijumlahkan per peran, hasilnya
menunjukkan porsi pengaruh setiap peran.

Kalau suatu peran mendominasi, artinya peran itulah yang paling menentukan skor
similarity, bukan peran lain yang mungkin lebih relevan secara makna.

Cell berikut memisahkan kolom matriks berdasarkan prefiks peran, lalu menjumlahkan
bobot kuadratnya per film.


In [17]:
ROLE_PREFIXES = ("genre_", "director_", "writer_", "cast_")

# get_feature_names_out() mengembalikan dtype object, bukan str NumPy.
# np.char.startswith hanya menerima dtype str, jadi harus dikonversi dulu.
vocabulary = np.asarray(vectorizer.get_feature_names_out(), dtype=str)

role_of_feature = np.full(len(vocabulary), "lainnya", dtype=object)
for prefix in ROLE_PREFIXES:
    mask = np.char.startswith(vocabulary, prefix)
    role_of_feature[mask] = prefix.rstrip("_")

role_share = {}

for prefix in ROLE_PREFIXES:
    role = prefix.rstrip("_")
    columns = np.flatnonzero(role_of_feature == role)
    share = np.asarray(matrix[:, columns].power(2).sum(axis=1)).ravel()
    role_share[role] = share

share_frame = pd.DataFrame(role_share)

print("jumlah share per film:", round(float(share_frame.sum(axis=1).median()), 6))

summary = pd.DataFrame(
    {
        "rata_rata_kontribusi": share_frame.mean().round(4),
        "median_kontribusi": share_frame.median().round(4),
        "persen_film_didominasi": (
            share_frame.idxmax(axis=1).value_counts(normalize=True) * 100
        ).round(2),
    }
)

summary

jumlah share per film: 1.0


,rata_rata_kontribusi,median_kontribusi,persen_film_didominasi
cast,0.4331,0.5000,72.22
director,0.1855,0.1250,7.71
genre,0.1845,0.1818,13.04
writer,0.1970,0.1818,7.04


Hasilnya sangat jelas, dan ini temuan utama bagian ini.

| Peran | Rata-rata kontribusi | Median | Persen film didominasi |
|---|---:|---:|---:|
| `cast` | 0,4331 | 0,5000 | 72,22 persen |
| `writer` | 0,1970 | 0,1818 | 7,04 persen |
| `director` | 0,1855 | 0,1250 | 7,71 persen |
| `genre` | 0,1845 | 0,1818 | 13,04 persen |

`cast` menyumbang rata-rata 43,31 persen bobot, dan mendominasi di 72,22 persen film.
Artinya untuk hampir tiga dari empat film, aktor adalah faktor penentu utama kemiripan
metadata, bukan sutradara.

Sutradara hanya menyumbang 18,55 persen, dengan median 12,5 persen. Ini sangat kecil,
padahal sutradara biasanya sinyal paling kuat untuk kemiripan film.

Genre menyumbang 18,45 persen walaupun hanya punya 19 fitur dari total 841.843. Ini
karena hampir semua film punya genre, sementara nama orang jarang berulang. Jadi genre
menang karena kedekatan, bukan karena jumlah fitur.

Jumlah share per film tepat 1,0. Ini memastikan perhitungannya benar, karena matriks
sudah dinormalisasi L2 dan total bobot kuadrat setiap film harus 1.

### Kenapa efek ini muncul

Setiap token diberi bobot yang sama, yaitu satu dibagi akar jumlah token film. Karena
`cast` punya hingga 5 token sementara `directors` hanya 1, aktor otomatis mendominasi.

Contoh nyata pada Inception yang punya 10 token:

```text
genre    : 3 token  -> 30,0 persen bobot
cast     : 5 token  -> 50,0 persen bobot
director : 1 token  -> 10,0 persen bobot
writer   : 1 token  -> 10,0 persen bobot
```

Sutradara, yang secara semantik paling menentukan, hanya punya 10 persen pengaruh.

### Efek normalisasi L2 pada skor

Selain dominasi peran, normalisasi L2 juga menghukum film dengan metadata banyak. Ini
efek matematis murni, bukan pilihan desain.

Inception punya 10 token. Blood and Steel punya 5 token. Keduanya berbagi 3 token yang
sama, yaitu tiga genre. Skor cosine-nya:

```text
3 / akar(10 x 5)  = 3 / 7,071  = 0,4243
```

The Dark Knight punya 14 token, dan berbagi 3 token yang sama dengan Inception,
termasuk sutradara dan penulis. Skor cosine-nya:

```text
3 / akar(10 x 14) = 3 / 11,832 = 0,2535
```

Jumlah token yang cocok sama, yaitu tiga. Tapi skor The Dark Knight lebih rendah,
karena penyebutnya lebih besar. Artinya film dengan metadata sedikit otomatis menang
untuk setiap kecocokan, walaupun isinya lebih tipis.


## 10. Bukti Vektor Metadata yang Identik

Bagian ini memeriksa satu masalah yang berbeda dari bagian 9. Kalau di bagian 9 yang
dibahas adalah peran mana yang paling berpengaruh, di sini yang dibahas adalah film
yang tidak punya cukup informasi untuk dibedakan sama sekali.

Sebagian film memiliki himpunan token metadata yang persis sama dengan film lain. Kalau
himpunan tokennya identik, vektornya juga identik setelah normalisasi L2, dan cosine
similarity menjadi 1,0.

Bagian ini hanya membaca data. Tidak ada baris yang dibuang atau diubah.


Cell berikut membangun "signature" untuk setiap film, yaitu daftar kolom matriks yang
nilainya bukan nol.

Dua film punya cosine similarity 1,0 jika dan hanya jika himpunan kolom aktifnya sama.
Jadi cukup membandingkan daftar indeks, tanpa perlu menghitung cosine penuh. Ini
penting karena menghitung seluruh pasangan berarti 561 GB, sementara daftar indeks
hanya butuh beberapa ratus MB.

Hasilnya 5.648 grup dengan vektor identik, melibatkan 19.037 film atau 4,92 persen
dataset. Artinya hampir satu dari dua puluh film tidak punya pembeda apa pun dari
film lain di jalur metadata.


In [18]:
import joblib
import numpy as np
import pandas as pd
from collections import Counter
from scipy import sparse

matrix = sparse.load_npz(METADATA_DIR / "metadata_matrix.npz").tocsr()
ids = np.load(METADATA_DIR / "movie_ids.npy")
vectorizer = joblib.load(METADATA_DIR / "vectorizer.joblib")
vocabulary = np.asarray(vectorizer.get_feature_names_out(), dtype=str)

# Signature sebuah film adalah daftar kolom yang bernilai bukan nol.
# Dua film dianggap identik kalau signature-nya sama persis.
signatures = [
    tuple(matrix.indices[matrix.indptr[i] : matrix.indptr[i + 1]])
    for i in range(matrix.shape[0])
]

signature_counts = Counter(signatures)
duplicate_groups = {
    sig: count for sig, count in signature_counts.items() if count > 1
}

films_in_duplicates = sum(duplicate_groups.values())

ringkasan = {
    "total film": len(ids),
    "grup vektor identik": len(duplicate_groups),
    "film dalam grup identik": films_in_duplicates,
    "persen dari dataset": round(films_in_duplicates / len(ids) * 100, 2),
}

pd.Series(ringkasan, name="value")

total film                 387188.00
grup vektor identik          5648.00
film dalam grup identik     19037.00
persen dari dataset             4.92
Name: value, dtype: float64

Cell berikut melihat sebaran ukuran grup. Tujuannya membedakan dua hal yang berbeda.

Grup kecil, berisi 2 atau 3 film, biasanya terjadi karena dua film memang punya
metadata yang sama persis. Ini wajar, terutama untuk film yang saling terkait.

Grup besar, berisi puluhan sampai ratusan film, menandakan masalah struktural. Itu
bukan kebetulan, melainkan tanda bahwa metadata film-film tersebut terlalu tipis untuk
membedakan apa pun.

Grup terbesar berisi 610 film, dan itu bukan kebetulan. Penjelasannya ada di cell
berikutnya.


In [19]:
ukuran_grup = Counter(duplicate_groups.values())

distribusi = pd.DataFrame(
    sorted(ukuran_grup.items()),
    columns=["film per grup", "jumlah grup"],
).set_index("film per grup")

distribusi["total film"] = distribusi.index * distribusi["jumlah grup"]

print("grup terbesar berisi:", distribusi.index.max(), "film")
print()
print(distribusi.head(8).to_string())
print("...")
print(distribusi.tail(4).to_string())

grup terbesar berisi: 610 film

               jumlah grup  total film
film per grup                         
2                     3751        7502
3                      880        2640
4                      352        1408
5                      201        1005
6                      103         618
7                       65         455
8                       66         528
9                       47         423
...
               jumlah grup  total film
film per grup                         
97                       1          97
110                      1         110
208                      1         208
610                      1         610


Cell berikut menampilkan lima grup terbesar beserta isinya. Ini bagian yang paling
meyakinkan saat dibawa ke bimbingan, karena memperlihatkan nama filmnya langsung.

Tiga pola yang muncul:

**Pola pertama, dokumenter tanpa metadata.** Grup 610 film semuanya hanya punya token
`genre_documentary`, tanpa sutradara, penulis, atau aktor. Contohnya `A Hand Shake`,
`Bassin des Tuileries`, dan `Attelage d'un camion`.

**Pola kedua, satu sutradara produktif.** Grup 208 film semuanya karya Stan Brakhage,
pembuat film eksperimental. Tokennya hanya `director_stan_brakhage`, tanpa genre
maupun aktor.

**Pola ketiga, kombinasi tipis.** Grup 110 film punya `genre_documentary` dan
`cast_james_a_fitzpatrick`, yang ternyata aktor dokumenter lama. Dua token saja, dan
keduanya sama untuk 110 film.

Jadi masalahnya bukan kesalahan parsing. Metadata film-film itu memang hanya sebanyak
itu.


In [20]:
movies_full = pd.read_parquet(
    MODELING_PARQUET,
    columns=["id", "title", "genres", "directors"],
).set_index("id")

terbesar = sorted(
    duplicate_groups.items(),
    key=lambda item: item[1],
    reverse=True,
)[:5]

for signature, jumlah in terbesar:
    token = [str(vocabulary[col]) for col in signature]

    print("=" * 88)
    print(f"grup {jumlah} film | token: {token}")
    print("-" * 88)

    ditampilkan = 0
    for index, sig in enumerate(signatures):
        if sig != signature:
            continue

        movie_id = ids[index]
        judul = str(movies_full.loc[movie_id, "title"])[:46]
        sutradara = str(movies_full.loc[movie_id, "directors"])[:26]

        print(f"  {movie_id:>8} | {judul:<46} | {sutradara}")
        ditampilkan += 1

        if ditampilkan == 4:
            break
    print()

grup 610 film | token: ['genre_documentary']
----------------------------------------------------------------------------------------
    105157 | A Hand Shake                                   | nan
    190575 | Bassin des Tuileries                           | nan
    195133 | Attelage d'un camion                           | nan
    193625 | Dresden, August-Brücke                         | nan

grup 208 film | token: ['director_stan_brakhage']
----------------------------------------------------------------------------------------
    126200 | Rage Net                                       | Stan Brakhage
    126216 | Comingled Containers                           | Stan Brakhage
    104407 | Glaze of Cathexis                              | Stan Brakhage
     97517 | The Wold Shadow                                | Stan Brakhage

grup 110 film | token: ['genre_documentary', 'cast_james_a_fitzpatrick']
------------------------------------------------------------------------------------

Cell berikut menghitung akar masalahnya: berapa banyak token yang dimiliki setiap film.

Hasilnya 7.223 film hanya punya 1 token, dan 91.450 film punya 5 token atau kurang.
Artinya sekitar 23,6 persen dataset berada di zona metadata sangat tipis.

Sebanyak 1.027 film token-nya hanya genre, tanpa nama orang sama sekali. Dari jumlah
itu, 610 film hanya punya `genre_documentary`.

Film dengan 1 token punya vektor yang isinya satu nilai 1,0 setelah normalisasi. Tidak
ada arah lain, tidak ada pembeda. Semua film dengan token yang sama persis akan
menghasilkan vektor yang sama persis.


In [21]:
nnz_per_film = np.diff(matrix.indptr)

role_mask = {
    role: np.char.startswith(vocabulary, role + "_")
    for role in ("genre", "director", "writer", "cast")
}

genre_only = 0
for index in range(matrix.shape[0]):
    kolom = matrix.indices[matrix.indptr[index] : matrix.indptr[index + 1]]
    if role_mask["genre"][kolom].all():
        genre_only += 1

doc_column = int(vectorizer.vocabulary_["genre_documentary"])
doc_only = 0
for index in range(matrix.shape[0]):
    awal = matrix.indptr[index]
    akhir = matrix.indptr[index + 1]
    if akhir - awal == 1 and matrix.indices[awal] == doc_column:
        doc_only += 1

ringkasan_token = {
    "film dengan 1 token": int((nnz_per_film == 1).sum()),
    "film dengan 2 token": int((nnz_per_film == 2).sum()),
    "film dengan 3 token": int((nnz_per_film == 3).sum()),
    "film dengan 5 token atau kurang": int((nnz_per_film <= 5).sum()),
    "film yang token-nya hanya genre": genre_only,
    "film yang token-nya hanya genre_documentary": doc_only,
    "persen film hanya genre": round(genre_only / matrix.shape[0] * 100, 2),
}

pd.Series(ringkasan_token, name="value")

film dengan 1 token                             7223.00
film dengan 2 token                            20739.00
film dengan 3 token                            23749.00
film dengan 5 token atau kurang                91450.00
film yang token-nya hanya genre                 1027.00
film yang token-nya hanya genre_documentary      610.00
persen film hanya genre                            0.27
Name: value, dtype: float64

### Temuan

Empat angka utama yang perlu dibawa ke bimbingan.

| Temuan | Nilai | Arti |
|---|---:|---|
| Grup vektor identik | 5.648 grup | Pasangan atau kelompok film tanpa pembeda apa pun |
| Film dalam grup identik | 19.037 (4,92 persen) | Hampir satu dari dua puluh film |
| Film dengan hanya 1 token | 7.223 | Vektor hanya berisi satu nilai 1, tanpa pembeda |
| Film dengan hanya `genre_documentary` | 610 | Vektor identik satu sama lain secara sempurna |

Akar masalahnya adalah ketiadaan metadata, bukan kesalahan parsing.

Konsekuensinya untuk fusion perlu dicatat. Film seperti `A Hand Shake` akan selalu
mendapat skor metadata 1,0 terhadap 609 film lain, sehingga hasil rekomendasinya
ditentukan oleh urutan indeks, bukan relevansi. Pada nilai `alpha` rendah, kondisi ini
akan mendominasi hasil.

Ini berbeda dari masalah di bagian 9. Di sana yang dibahas adalah peran mana yang
paling berpengaruh. Di sini yang dibahas adalah film yang tidak punya cukup informasi
untuk dibedakan sama sekali.


## 11. Pilihan Penanganan

Ada dua masalah berbeda yang sudah terukur di notebook ini, dan masing-masing butuh
keputusan sendiri.

**Masalah pertama, dominasi peran.** `cast` menyumbang 43,31 persen bobot dan
mendominasi 72,22 persen film. Ini bisa dikurangi dengan menurunkan `CAST_LIMIT`.

**Masalah kedua, vektor identik.** 19.037 film punya vektor yang sama persis dengan
film lain. Ini tidak bisa diselesaikan dengan mengubah `CAST_LIMIT`, karena penyebabnya
adalah metadata yang memang kosong.

Untuk masalah pertama, tiga pilihan:

| Pilihan | Efek | Konsekuensi |
|---|---|---|
| `CAST_LIMIT = 5` (sekarang) | Cast mendominasi 72,22 persen film | Actor-driven, sutradara kalah |
| `CAST_LIMIT = 3` | Cast turun ke sekitar 35 persen | Informasi aktor berkurang |
| Bobot per peran | Paling langsung menyasar masalah | Angka bobot arbitrer, perlu analisis sensitivitas |

Untuk masalah kedua, tiga pilihan:

| Pilihan | Efek | Konsekuensi |
|---|---|---|
| Biarkan dan laporkan | Hasil sudah ada | Rekomendasi tidak deterministik pada alpha rendah |
| Buang film dengan 1 token | 7.223 film hilang | Dataset turun ke 379.965, encoding semantik harus diulang |
| Buang film dengan 2 token atau kurang | 27.962 film hilang | Lebih bersih, tapi pengurangan lebih besar |

Untuk tahap ini, notebook hanya melaporkan kedua masalah tanpa mengubah artifact.
Alasannya, setiap perubahan menuntut perhitungan ulang encoding semantik, top-K, dan
metadata. Keputusan itu sebaiknya diambil setelah seluruh pipeline selesai, supaya
tidak ada pekerjaan yang diulang lebih dari sekali.

Satu hal yang perlu ditegaskan: masalah vektor identik tidak muncul kalau `alpha`
tinggi, karena skor teks akan mendominasi. Masalah ini baru terasa ketika metadata
diberi bobot besar.


## Ringkasan Notebook

Notebook ini menghasilkan representasi metadata untuk 387.188 film dalam bentuk
matriks sparse 387.188 x 841.843 dengan 3.044.037 nilai bukan nol.

Angka kunci yang bisa dikutip:

| Item | Nilai |
|---|---|
| Film | 387.188 |
| Fitur unik | 841.843 |
| Nilai bukan nol | 3.044.037 |
| Kepadatan | 9,34 kali 10 pangkat -6 |
| Token per film (median) | 8 |
| Token per film (maksimum) | 575 |
| Norma setiap baris | 1,0 |
| Baris bernorma nol | 0 |
| Skor top-10 metadata | 0,36 sampai 0,46 |

Kontribusi per peran:

| Peran | Rata-rata | Dominan di |
|---|---:|---:|
| `cast` | 43,31 persen | 72,22 persen film |
| `writer` | 19,70 persen | 7,04 persen film |
| `director` | 18,55 persen | 7,71 persen film |
| `genre` | 18,45 persen | 13,04 persen film |

Dua keterbatasan terukur:

| Keterbatasan | Nilai |
|---|---:|
| Film dengan vektor identik | 19.037 (4,92 persen) |
| Film dengan hanya 1 token | 7.223 |
| Film dengan hanya `genre_documentary` | 610 |

Skor metadata top-10 berada di 0,36 sampai 0,46, sebanding dengan skor semantik di
0,52 sampai 0,65. Artinya kedua jalur cukup sebanding untuk fusion berbobot.

Yang belum dikerjakan dan menjadi tahap berikutnya: cosine similarity untuk seluruh
film pada jalur metadata, late fusion dengan parameter `alpha`, recommendation engine,
serta metrik evaluasi.

Satu catatan metodologis yang perlu dibawa: representasi ini memakai bobot seragam,
sehingga `cast` mendominasi 72,22 persen film. Selain itu, 4,92 persen film punya
vektor yang identik dengan film lain, karena metadatanya terlalu tipis. Keduanya
dilaporkan sebagai keterbatasan, bukan dikoreksi, agar jumlah variabel penelitian tetap
terkendali.
